# Study 818 — Trend Factor — the teardown

The headline spread's Newey-West *t*, the pooled Welch book test, the single-MA and momentum contrast, the 1,000-permutation placebo, the two-era robustness cut, the costed timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_names': 50, 'n_days': 3697, 'spread_bps': 1.42, 't_nw': 0.99, 't_1s': 0.87, 'lo_bps': 7.06, 'hi_bps': 8.48, 'welch_t': 0.49, 'gross_sharpe': 0.23, 'ma200_bps': 2.01, 'ma200_t': 1.24, 'mom_bps': 2.24, 'mom_t': 1.4, 'placebo_obs': 1.42, 'placebo_mean': 0.019, 'placebo_sd': 0.964, 'placebo_p': 0.066, 'placebo_sigma': 1.45, 'placebo_draws': 1000, 'era_early_bps': 1.72, 'era_early_t': 0.96, 'era_early_n': 1563, 'era_late_bps': 1.2, 'era_late_t': 0.57, 'era_late_n': 2134, 'timer_1_gross': 1.42, 'timer_1_cost': 2.14, 'timer_1_net': -0.72, 'timer_1_t': -0.44, 'timer_5_gross': 1.42, 'timer_5_cost': 10.14, 'timer_5_net': -8.72, 'timer_5_t': -5.34, 'null_mean_t': 0.07, 'null_sd_t': 0.94, 'null_fire': 1, 'planted_t': 10.28, 'planted_welch': 10.64}

## The headline — long-high-trend / short-low-trend spread

Daily equal-weight top-30% minus bottom-30% trend-factor spread.

In [2]:
print(f"spread        : {R['spread_bps']:+.2f} bps/day  NW(10) t = {R['t_nw']:+.2f}  "
      f"one-sample t = {R['t_1s']:+.2f}")
print(f"books         : high-trend {R['hi_bps']:+.2f} vs low-trend {R['lo_bps']:+.2f} bps "
      f"(Welch t = {R['welch_t']:+.2f})")
print(f"gross Sharpe  : {R['gross_sharpe']:+.2f} (before cost)")

spread        : +1.42 bps/day  NW(10) t = +0.99  one-sample t = +0.87
books         : high-trend +8.48 vs low-trend +7.06 bps (Welch t = +0.49)
gross Sharpe  : +0.23 (before cost)


## Contrast — the two sorts the trend factor is *claimed to beat*

The paper's central claim is that the blend dominates single-MA timing and momentum. Here it is the weakest of the three.

In [3]:
print(f"trend factor (blend) : {R['spread_bps']:+.2f} bps  NW t = {R['t_nw']:+.2f}")
print(f"single-MA(200) timing: {R['ma200_bps']:+.2f} bps  NW t = {R['ma200_t']:+.2f}")
print(f"12-1 momentum        : {R['mom_bps']:+.2f} bps  NW t = {R['mom_t']:+.2f}")

trend factor (blend) : +1.42 bps  NW t = +0.99
single-MA(200) timing: +2.01 bps  NW t = +1.24
12-1 momentum        : +2.24 bps  NW t = +1.40


## Placebo — column-permute the forward returns (1,000 permutations)

In [4]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> p = {R['placebo_p']:.3f} ({R['placebo_sigma']:+.2f} sd from mean)")

observed +1.42 bps vs placebo mean +0.019 (sd 0.964) -> p = 0.066 (+1.45 sd from mean)


## Robustness — two eras (split 2018-01-01)

In [5]:
print(f"2010-2017 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps  NW t = {R['era_early_t']:+.2f}")
print(f"2018-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps  NW t = {R['era_late_t']:+.2f}")

2010-2017 (n=1563): +1.72 bps  NW t = +0.96
2018-2026 (n=2134): +1.20 bps  NW t = +0.57


## The timer — can you get paid for it?

2 sides × one-way cost × NAV per day on the long-short book; short pays 50 bps/yr borrow.

In [6]:
for tag,g,c,n,t in [('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_net'],R['timer_1_t']),
                    ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_net'],R['timer_5_t'])]:
    print(f"{tag:>5} one-way: gross {g:+.2f} -> net {n:+.2f} bps/day (cost {c:.2f}/day, t={t:+.2f})")

 1 bp one-way: gross +1.42 -> net -0.72 bps/day (cost 2.14/day, t=-0.44)
5 bps one-way: gross +1.42 -> net -8.72 bps/day (cost 10.14/day, t=-5.34)


## Synthetic positive control — the machinery is unbiased

Live: the fitted trend factor must NOT fire on the null and must recover a planted relation.

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from trend_factor import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=818+s, n_assets=40, n_days=1500), beta_window=120)['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0015, seed=818, n_assets=40, n_days=1500), beta_window=120)
print(f"planted (edge=0.0015): NW t = {planted['t_nw']:+.2f}, Welch t = {planted['welch_t']:+.2f}")

null (edge=0), 8 seeds: NW t mean -0.08 (sd 0.71), |t|>=2 in 0/8


planted (edge=0.0015): NW t = +10.28, Welch t = +10.64


## Verdict

- **Signal — None.** The claimed Han-Zhou-Zhu trend factor does **not** replicate on 50 liquid US mega-caps: the long-high-trend / short-low-trend spread is **+1.42 bps/day** (NW *t* = **+0.99**) — the right sign but indistinguishable from zero, only ~1.45 sd into the permutation null (p = 0.066), weak in both eras (*t* = +0.96 / +0.57), and — fatally for the claim — **weaker than the single-MA(200) (*t* = +1.24) and momentum (*t* = +1.40) sorts it is supposed to beat**. The 20-seed synthetic control recovers a *planted* relation cleanly (*t* = +10.28, fires on 1/20 nulls), so this is a true null, not machinery. Survivorship biases the magnitude upward.
- **Tradability — Mirage.** The +1.42 bps/day gross edge is smaller than the 2.14 bps/day round-trip friction at a mere 1 bp one-way, so the book is net **-0.72 bps/day** (*t* = -0.44); at 5 bps **-8.72 bps/day**.